# Лабораторна робота № 5
## Географічні дані OpenStreetMap: від тегів до маршруту

**Інтелектуальний аналіз та візуалізація даних** · 4 години · 4 бали

---

### Постановка

Ви — аналітик у туристичному застосунку. Завдання: для заданого міста
**відібрати десять найвартісніших історико-культурних об'єктів** і
**спланувати піший маршрут**, який починається на залізничному вокзалі,
обходить усі десять і повертається на вокзал.

Слово «найвартісніші» в даних не записане. Критерій ви побудуєте самі —
і захистите його.

Потім те саме для **тридцяти** об'єктів. Результат цієї, більшої задачі
стане базою порівняння в **лабораторній роботі № 8**, де ту саму задачу
розв'язуватимуть генетичним та мурашиним алгоритмами.

### Що здаєте

1. Цей ноутбук із виконаними комірками (усі виводи збережені).
2. Файл `lr5_результати.json` — результати для ЛР8.
3. Карту `lr5_маршрут.html`.
4. Звіт у кінці ноутбука: відповіді на сім питань.

### Що оцінюється

| критерій | 0 | 2 | 3 | 4 |
|---|---|---|---|---|
| Відбір об'єктів | теги взято навмання | фільтр працює | фільтр обґрунтовано | показано, що відкинуто і чому |
| Критерій рейтингу | немає | є формула | ваги пояснено | порівняно з альтернативним критерієм |
| Маршрут | не будується | будується | перевірено коректність | пояснено різницю методів числом |
| Карта | немає | точки на карті | маршрут по ребрах графа | видно різницю з прямими відрізками |
| Звіт | немає | формально | з числами | з висновком, якого не було в завданні |

---
## 0. Ваш варіант

Впишіть прізвище та номер у списку групи. Місто підтягнеться само.

In [ ]:
STUDENT_NAME = 'Бубельник Юрій'
LIST_NUMBER  = 2

CITIES = [
    'Львів', 'Київ', 'Харків', 'Одеса', 'Дніпро',
    'Чернівці', 'Івано-Франківськ', 'Тернопіль', 'Ужгород', 'Луцьк',
    'Рівне', 'Хмельницький', 'Вінниця', 'Житомир', 'Чернігів',
    'Черкаси', 'Полтава', 'Суми', 'Кропивницький', 'Запоріжжя',
    'Миколаїв', 'Кривий Ріг', 'Кам’янець-Подільський', 'Мукачево',
    'Коломия', 'Дрогобич', 'Біла Церква', 'Ніжин', 'Остріг', 'Умань',
]

assert STUDENT_NAME and LIST_NUMBER > 0, 'впишіть прізвище та номер у списку'
CITY = CITIES[(LIST_NUMBER - 1) % len(CITIES)] + ', Україна'
print('варіант %d: %s' % (LIST_NUMBER, CITY))

# Якщо викладач призначив інше місто — впишіть, напр. 'Львів, Україна'. None — за варіантом.
CITY_OVERRIDE = None
if CITY_OVERRIDE:
    CITY = CITY_OVERRIDE
    print('місто замінено вручну:', CITY)

> Якщо викладач призначив вам інше місто — просто перепишіть `CITY` вручну
> і зазначте це у звіті.

In [ ]:
import importlib.util, subprocess, sys
REQUIRED = ['osmnx', 'geopandas', 'shapely', 'pyproj', 'pyarrow',
            'folium', 'mapclassify', 'h3', 'networkx']
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print('встановлюю:', ', '.join(missing), '...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('усі пакети на місці')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json, time, os, collections
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import folium
from xyzservices import providers as xyz

BASEMAP = xyz.Esri.WorldGrayCanvas   # CARTO з 2026 р. вимагає ключ API
pd.set_option('display.width', 200)
print('geopandas', gpd.__version__)

---
## 1. Два способи дістати дані

Робоча програма вимагає порівняти два способи доступу **за обсягом переданих
даних і за часом**:

* **живий запит** — `osmnx` питає Overpass API просто зараз;
* **знімок** — ті самі дані, збережені у файли GeoParquet, читаються з диска.

Нижче — живий запит. Запустіть його **один раз**: він збереже теку зі знімком,
і всі наступні запуски читатимуть з неї.

> Overpass — безкоштовний сервіс і обмежує частоту. Якщо запит не пройшов —
> зачекайте хвилину й повторіть. Якщо не проходить взагалі, візьміть
> резервний знімок у викладача і вкажіть це у звіті.

In [ ]:
import osmnx as ox

OUT = 'osm_' + CITY.split(',')[0].strip().lower().replace(' ', '_')
os.makedirs(OUT, exist_ok=True)
ox.settings.use_cache = True
ox.settings.cache_folder = os.path.join(OUT, 'cache')
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True

TOURISM_OK = ['museum', 'attraction', 'gallery', 'artwork', 'viewpoint']

def cache_bytes():
    import pathlib
    p = pathlib.Path(ox.settings.cache_folder)
    return sum(f.stat().st_size for f in p.glob('*.json')) if p.exists() else 0

print('тека знімка:', OUT)

### Дві налаштовані дрібниці, без яких запит не пройде

Це не формальності — кожна з них ламає збір по-своєму, і жодна не схожа
на свою справжню причину.

**Представитись.** Overpass відповідає **HTTP 406** клієнтові, який прийшов
із типовим заголовком `python-requests`. `osmnx` надсилає свій `User-Agent`
сам, але якщо ви звертаєтесь до сервісу напряму — надсилайте теж.
Рядок має бути **лише латиницею**: HTTP-заголовки передаються в latin-1,
і кирилиця кладе запит помилкою `UnicodeEncodeError` ще до відправлення.

**Обрати машину.** За іменем `overpass-api.de` стоїть кілька серверів, і
частина з них відмовляє у з’єднанні. `osmnx` закріплює одну адресу на
всю сесію — навмисно, щоб не порушувати облік слотів. Якщо закріпиться та,
що відмовляє, впаде весь збір. Тому адресу обираємо самі, вже перевірену.

In [ ]:
import socket

USER_AGENT = ('LvivPolytechnic-SK26-coursework/1.0 '
              '(https://lpnu.ua; student lab assignment)')
assert USER_AGENT.isascii(), 'HTTP-заголовки — latin-1, кирилиця тут не працює'
ox.settings.http_user_agent = USER_AGENT


def pin_working_ip(hostname, port=443, timeout=8):
    """Закріпити для цього імені адресу, яка справді відповідає."""
    try:
        infos = socket.getaddrinfo(hostname, port, socket.AF_INET, socket.SOCK_STREAM)
    except Exception as e:
        print('не вдалось резолвити %s: %s' % (hostname, type(e).__name__))
        return None
    ips = sorted({i[4][0] for i in infos})
    good = []
    for ip in ips:
        s = socket.socket(); s.settimeout(timeout)
        try:
            s.connect((ip, port)); good.append(ip)
        except Exception:
            pass
        finally:
            s.close()
    print('%s: машин %d, відповідають %d %s' % (hostname, len(ips), len(good), good))
    if not good:
        return None
    original = socket.gethostbyname
    socket.gethostbyname = lambda h: good[0] if h == hostname else original(h)
    print('закріплено', good[0])
    return good[0]


pin_working_ip('overpass-api.de')

> Якщо жодна машина не відповіла — виконайте комірку ще раз за хвилину,
> або візьміть у викладача резервний знімок і зазначте це у звіті.
> Інші дзеркала перелічені на сторінці
> [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API);
> адресу можна підставити у `ox.settings.overpass_url`.

### Завдання 1.1

Допишіть функцію `fetch`: вона має зберегти шар у `.parquet`, а якщо файл
уже є — прочитати його, **не звертаючись до мережі**. Поверніть також,
скільки МБ пішло мережею і скільки секунд це зайняло.

Підказка щодо `clean`: у OSM одна колонка може містити і число, і список
(ребро, склеєне з кількох шляхів). Parquet вимагає один тип на колонку,
тому неоднорідну колонку треба звести до тексту **цілком**.

In [ ]:
def clean(gdf):
    g = gdf.reset_index()
    for c in g.columns:
        if c == 'geometry':
            continue
        boxed = g[c].map(lambda v: isinstance(v, (list, dict, set, tuple))).any()
        if boxed or g[c].dtype == object:
            g[c] = g[c].map(
                lambda v: None if v is None or (isinstance(v, float) and pd.isna(v))
                else (json.dumps(v, ensure_ascii=False)
                      if isinstance(v, (list, dict, set, tuple)) else str(v))
            ).astype('string')
    return g


COST = {}

def fetch(name, getter):
    """Повертає шар. Якщо файл уже збережено — читає з диска."""
    path = os.path.join(OUT, name + '.parquet')
    if os.path.exists(path) and os.path.getsize(path) > 0:
        t0 = time.time()
        g = gpd.read_parquet(path)
        COST[name] = {'спосіб': 'знімок', 'записів': len(g),
                      'передано_МБ': 0.0,
                      'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                      'секунд': round(time.time() - t0, 3)}
        return g

    # живий запит: міряємо приріст кешу osmnx (= скільки прийшло мережею) і час
    t0, b0 = time.time(), cache_bytes()
    g = clean(getter())
    g.to_parquet(path, index=False)
    g = gpd.read_parquet(path)          # читаємо назад, щоб типи були як у знімку
    COST[name] = {'спосіб': 'живий запит', 'записів': len(g),
                  'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                  'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                  'секунд': round(time.time() - t0, 1)}
    return g

In [ ]:
boundary = fetch('boundary', lambda: ox.geocode_to_gdf(CITY))
poly = boundary.geometry.iloc[0]
print('межа міста отримана, площа %.1f км²'
      % (boundary.to_crs(32635).area.iloc[0] / 1e6))

In [ ]:
heritage = fetch('heritage', lambda: ox.features_from_polygon(
    poly, {'historic': True, 'tourism': TOURISM_OK}))
stations = fetch('stations', lambda: ox.features_from_polygon(
    poly, {'railway': ['station', 'halt'], 'building': ['train_station']}))
amenities = fetch('amenities', lambda: ox.features_from_polygon(
    poly, {'amenity': ['cafe', 'restaurant', 'drinking_water', 'toilets']}))

# Мережа приходить двома таблицями одразу, тому їй окрема гілка.
NODES_P = os.path.join(OUT, 'road_nodes.parquet')
EDGES_P = os.path.join(OUT, 'road_edges.parquet')

if os.path.exists(NODES_P) and os.path.exists(EDGES_P):
    t0 = time.time()
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'знімок', 'записів': len(nodes) + len(edges),
                           'передано_МБ': 0.0,
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 3)}
else:
    t0, b0 = time.time(), cache_bytes()
    G_raw = ox.graph_from_polygon(poly, network_type='walk')
    nodes, edges = ox.convert.graph_to_gdfs(G_raw)
    clean(nodes).to_parquet(NODES_P, index=False)
    clean(edges).to_parquet(EDGES_P, index=False)
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'живий запит', 'записів': len(nodes) + len(edges),
                           'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 1)}

print('вузлів %d | ребер %d' % (len(nodes), len(edges)))

### Завдання 1.2

Зведіть `COST` у таблицю і дайте відповідь у звіті: **у скільки разів
читання знімка швидше за живий запит** і **скільки мегабайтів ви не передали
мережею** при повторному запуску.

Контрольна ознака правильності: при **другому** запуску ноутбука всі рядки
мають мати спосіб «знімок», а `передано_МБ` — нуль.

In [ ]:
cost = pd.DataFrame(COST).T
display(cost)

LIVE_P = os.path.join(OUT, 'cost_live.json')
live = cost[cost['спосіб'] == 'живий запит']
t_now = cost['секунд'].astype(float).sum()
if len(live):
    # перший запуск: запам'ятовуємо ціну живого запиту, щоб порівняти на другому
    json.dump({'секунд': float(live['секунд'].astype(float).sum()),
               'передано_МБ': float(live['передано_МБ'].astype(float).sum())},
              open(LIVE_P, 'w', encoding='utf-8'), ensure_ascii=False)
    TIMING = ('живий запит %.1f с, %.1f МБ мережею. ПЕРЕЗАПУСТІТЬ ноутбук (Restart & Run All), '
              'щоб отримати порівняння зі знімком.' % (live['секунд'].astype(float).sum(),
                                                       live['передано_МБ'].astype(float).sum()))
elif os.path.exists(LIVE_P):
    lv = json.load(open(LIVE_P, encoding='utf-8'))
    TIMING = ('живий запит %.1f с і %.1f МБ мережею; знімок %.2f с і 0 МБ — '
              'у %.0f разів швидше, %.1f МБ не передано повторно.'
              % (lv['секунд'], lv['передано_МБ'], t_now, lv['секунд'] / max(t_now, 1e-3), lv['передано_МБ']))
else:
    TIMING = 'усе прочитано зі знімка за %.2f с, 0 МБ мережею (живого запуску в цій теці не було).' % t_now
print(TIMING)


### Перевірка: чи справді мережа пішохідна

Тип мережі не записаний у самих файлах — його видно лише з тегів `highway`.
У пішохідній мережі мають бути `footway`, `pedestrian`, `path`, `steps`.

In [ ]:

FOOT = {'footway', 'pedestrian', 'path', 'steps', 'living_street', 'track', 'corridor'}
hw = edges['highway'].astype(str)
is_foot = hw.map(lambda s: any(f in s for f in FOOT - {'living_street', 'track'}))
print('ребер усього            :', len(edges))
print('з них footway/pedestrian/path/steps: %d (%.1f%%)' % (is_foot.sum(), 100 * is_foot.mean()))
print('частка односторонніх    : %.1f%%' % (100 * edges['oneway'].astype(str).str.lower().eq('true').mean()))
print(hw.value_counts().head(8).to_string())
WALK_OK = is_foot.mean() > 0.05
if not WALK_OK:
    print('\nУВАГА: пішохідних ребер немає — це автомобільна мережа (network_type="drive").')
    print('Очікуйте злиплі точки на кроці 4 та завищений коефіцієнт обходу на кроці 5.')


> **Чому пішохідна мережа, а не автомобільна.** У старому місті музеї стоять
> за сотню метрів один від одного, а автомобільних вузлів там одиниці. Кілька
> пам'яток сядуть на **один** вузол графа, і матриця віддалей скаже, що між
> ними нуль метрів. Задача розв'яжеться, відповідь буде неправильною, і
> повідомлення про помилку не з'явиться. У кроці 4 ви це перевірите явно.

---
## 2. Хто претендує на звання пам'ятки

У OpenStreetMap немає тега «визначна пам'ятка». Є два сімейства:

* `historic=*` — `castle`, `church`, `monument`, `memorial`, `ruins`, `tower`, `yes`…
* `tourism=*` — `museum`, `gallery`, `attraction`, `artwork`, `viewpoint`

Перетин неповний: музей у сучасній будівлі має лише `tourism`, руїни фортеці —
лише `historic`. Тому беремо **об'єднання**.

📖 <https://wiki.openstreetmap.org/wiki/Map_Features>

### Завдання 2.1

Відберіть кандидатів і **назвіть у звіті, скільки об'єктів ви відкинули й чому**.

In [ ]:
def col(df, c):
    """Колонка або порожня серія, якщо такого тега в шарі немає взагалі."""
    return df[c] if c in df.columns else pd.Series(pd.NA, index=df.index, dtype='string')

hist = col(heritage, 'historic')
tour = col(heritage, 'tourism')
name = col(heritage, 'name')

is_hist = hist.notna() & ~hist.isin(['no']).fillna(False)
is_tour = tour.isin(TOURISM_OK).fillna(False)
mask = (is_hist | is_tour).fillna(False)

# Не цілі для пішої екскурсії: дрібні придорожні об'єкти і меморіальні дошки
SMALL = ['boundary_stone', 'milestone', 'wayside_cross', 'wayside_shrine', 'tomb', 'grave']
small = hist.isin(SMALL).fillna(False) & ~is_tour
plaque = col(heritage, 'memorial').isin(['plaque', 'stolperstein', 'blue_plaque']).fillna(False) & ~is_tour
has_name = name.notna() & (name.astype('string').str.strip() != '').fillna(False)

cand = heritage[mask & ~small & ~plaque & has_name].reset_index(drop=True)

print('усього у шарі heritage :', len(heritage))
print('підходять за тегами    :', int(mask.sum()), ' (відкинуто тегом: %d, напр. tourism=hotel)' % int((~mask).sum()))
print('  - придорожні/надгробки:', int((mask & small).sum()), SMALL)
print('  - меморіальні дошки   :', int((mask & plaque).sum()))
print('з них мають name       :', len(cand), ' (без назви відкинуто: %d)' % int((mask & ~small & ~plaque & ~has_name).sum()))

DROPPED = {
    'не той тег': int((~mask).sum()),
    'дрібні об’єкти': int((mask & small).sum()),
    'меморіальні дошки': int((mask & plaque).sum()),
    'без назви': int((mask & ~small & ~plaque & ~has_name).sum()),
}
print()
print('що саме без назви (historic/tourism):')
print(pd.concat([hist, tour], axis=1)[mask & ~has_name].fillna('—').value_counts().head(8).to_string())

In [ ]:
assert 'cand' in dir(), 'спершу виконайте завдання 2.1'
print('=== historic ===');  print(cand['historic'].value_counts().head(10).to_string())
print()
print('=== tourism ===');   print(cand['tourism'].value_counts().head(10).to_string())
print()
print('типи геометрії:', cand.geometry.geom_type.value_counts().to_dict())

> **Перевірте себе.** Серед типів геометрії мають бути і `Point`, і `Polygon`.
> Якщо самі лише полігони — ви відфільтрували зайве: пам'ятники, меморіали та
> оглядові точки в OSM є **вузлами**, у них немає будівлі.

### Якщо кандидатів менше тридцяти

Невелике місто може дати 15–20 об'єктів. Тоді:

1. розширте перелік `tourism` (наприклад, додайте `theatre`, `place_of_worship`);
2. або зменште `N_LARGE` до того, що є;

і **обов'язково зазначте це у звіті** разом із числом.

---
## 3. Критерій «найкращі»

Ключовий крок роботи. Стовпця «цінність» у даних немає. Його треба
сконструювати і захистити.

Сигнали, які є в OSM:

| сигнал | що означає |
|---|---|
| `wikidata` / `wikipedia` | про об'єкт є стаття — хтось сторонній визнав його вартим опису |
| кількість колонок `name:*` | назву переклали N мовами — перекладають те, що шукають іноземці |
| `tourism=museum/gallery` | призначений для відвідування |
| `historic` заповнений | має історичну природу |
| площа полігона | розмір — **найслабший сигнал**: великим буває і склад |

### Завдання 3.1

Побудуйте числовий рейтинг. Вимоги:

* використати **щонайменше три** різні сигнали;
* звести різнорідні величини до спільної шкали (підказка: `rank(pct=True)`);
* **пояснити у звіті, чому саме такі ваги**.

In [ ]:
metric = cand.to_crs(32635)
cand['area_m2'] = np.where(metric.geometry.geom_type.isin(['Polygon', 'MultiPolygon']),
                           metric.area.values, 0.0)

name_cols = [c for c in cand.columns if c.startswith('name:')]
cand['wiki']   = (col(cand, 'wikidata').notna() | col(cand, 'wikipedia').notna()).astype(int)
cand['n_lang'] = cand[name_cols].notna().sum(axis=1) if name_cols else 0
cand['visit']  = col(cand, 'tourism').isin(['museum', 'gallery']).fillna(False).astype(int)
cand['hist']   = col(cand, 'historic').notna().astype(int)

# Двійкові сигнали вже на шкалі 0..1; лічильники й площу — у ранги-перцентилі,
# щоб 400 000 м² парку не задавили 12 перекладів назви.
WEIGHTS = {'wiki': 0.35, 'n_lang': 0.30, 'visit': 0.15, 'hist': 0.10, 'area': 0.10}

def score_of(df, w):
    return (w['wiki']   * df['wiki']
          + w['n_lang'] * df['n_lang'].rank(pct=True)
          + w['visit']  * df['visit']
          + w['hist']   * df['hist']
          + w['area']   * df['area_m2'].rank(pct=True))

cand['score'] = score_of(cand, WEIGHTS)

print('кандидатів   :', len(cand))
print('мають wiki   :', int(cand['wiki'].sum()))
print('мають ≥1 name:* :', int((cand['n_lang'] > 0).sum()), '| колонок name:* у шарі:', len(name_cols))
print('музеї/галереї:', int(cand['visit'].sum()), '| historic:', int(cand['hist'].sum()))
print('кореляція Спірмена між сигналами:')
print(cand[['wiki', 'n_lang', 'visit', 'hist', 'area_m2']].corr(method='spearman').round(2).to_string())

### Завдання 3.2 — обов'язкове порівняння

Побудуйте **другий** рейтинг за найпростішим критерієм — площею будівлі —
і порахуйте, скільки об'єктів збігається в перших десяти.

Це головне питання роботи: обидва списки отримані з тих самих даних, тим
самим кодом, без жодної помилки. Якщо вони різні — поясніть у звіті, що це
означає для слова «найкращі».

In [ ]:
N_SMALL, N_LARGE = 10, 30
if len(cand) < N_LARGE:
    print('УВАГА: кандидатів лише %d, N_LARGE зменшено' % len(cand))
    N_LARGE = len(cand)

top_score = cand.nlargest(N_LARGE, 'score').reset_index().rename(columns={'index': 'cand_idx'})
top10     = top_score.head(N_SMALL)
top_naive = cand.nlargest(N_SMALL, 'area_m2').reset_index().rename(columns={'index': 'cand_idx'})
overlap   = len(set(top10['cand_idx']) & set(top_naive['cand_idx']))

print('перетин: %d з %d' % (overlap, N_SMALL))
display(pd.DataFrame({'за вашим критерієм': list(top10['name']),
                      'score': top10['score'].round(3).values,
                      'за площею': list(top_naive['name']),
                      'площа, м²': top_naive['area_m2'].round(0).astype(int).values}))

# Стійкість: як сильно топ-10 залежить від конкретних ваг?
rng = np.random.default_rng(42)
base = set(top10['cand_idx'])
stab = []
for _ in range(300):
    w = {k: v * rng.uniform(0.5, 1.5) for k, v in WEIGHTS.items()}
    s = score_of(cand, w)
    stab.append(len(base & set(s.nlargest(N_SMALL).index)))
STAB = np.array(stab)
print('ваги ±50%%, 300 спроб: збіг із базовим топ-10 у середньому %.1f, мінімум %d'
      % (STAB.mean(), STAB.min()))

### Завдання 3.3 — дублікати

OSM наповнюють різні люди. Той самий музей трапляється двічі під різними
назвами, а вежа буває окремим об'єктом усередині палацу. Знайдіть у `top_score`
пари, що стоять ближче **60 метрів** одна до одної, і вирішіть, що з ними робити.

Рішення (лишити / злити / видалити) опишіть у звіті.

In [ ]:
def close_pairs(gdf, limit_m=60):
    p = gdf.to_crs(32635).geometry.representative_point()
    XY = np.c_[p.x.values, p.y.values]
    D = np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    out = [(i, j, D[i, j]) for i in range(len(XY)) for j in range(i + 1, len(XY))
           if D[i, j] < limit_m]
    return sorted(out, key=lambda t: t[2])

import re
def norm(s):
    return re.sub(r'[^0-9a-zа-яіїєґ]', '', str(s).lower())

STOP = {'львівський', 'львівська', 'київський', 'київська', 'національний',
        'національна', 'державний', 'державна', 'музей', 'пам\u2019ятник', "пам'ятник"}

def is_duplicate(a, b):
    """Правило злиття: один об'єкт, записаний двічі, або вкладений в інший."""
    wa, wb = a.get('wikidata'), b.get('wikidata')
    if pd.notna(wa) and pd.notna(wb) and wa == wb:
        return 'та сама wikidata'
    if norm(a['name']) == norm(b['name']):
        return 'та сама назва'
    ta = set(re.findall(r'[0-9a-zа-яіїєґ]{3,}', str(a['name']).lower())) - STOP
    tb = set(re.findall(r'[0-9a-zа-яіїєґ]{3,}', str(b['name']).lower())) - STOP
    if ta and tb and len(ta & tb) / len(ta | tb) >= 0.6:
        return 'схожі назви (%d/%d слів)' % (len(ta & tb), len(ta | tb))
    ga, gb = a.geometry, b.geometry
    if ga.contains(gb.representative_point()) or gb.contains(ga.representative_point()):
        return 'вкладений'
    return None

DUPES = []          # (лишили, прибрали, відстань, причина)
NEAR_KEPT = []      # близькі, але різні об'єкти
work = cand.copy()
for it in range(10):
    ts = work.nlargest(N_LARGE, 'score').reset_index().rename(columns={'index': 'cand_idx'})
    drop = set()
    for i, j, d in close_pairs(ts):
        a, b = ts.iloc[i], ts.iloc[j]               # a має вищий score
        why = is_duplicate(a, b)
        if why and a['cand_idx'] not in drop:
            drop.add(b['cand_idx'])
            DUPES.append((a['name'], b['name'], round(d), why))
        elif not why and (a['name'], b['name']) not in [(x[0], x[1]) for x in NEAR_KEPT]:
            NEAR_KEPT.append((a['name'], b['name'], round(d)))
    if not drop:
        break
    work = work.drop(index=list(drop))

print('злито/прибрано дублікатів:', len(DUPES))
for k, r, d, why in DUPES:
    print('  %4d м  лишили «%s», прибрали «%s»  (%s)' % (d, k, r, why))
removed = {r for _, r, _, _ in DUPES}
NEAR_KEPT = [t for t in NEAR_KEPT if t[0] not in removed and t[1] not in removed]
print('близькі (<60 м), але різні — лишили обидва:', len(NEAR_KEPT))
for a, b, d in NEAR_KEPT:
    print('  %4d м  «%s» — «%s»' % (d, a, b))

cand_raw = cand
cand = work.reset_index(drop=True)
top_score = cand.nlargest(N_LARGE, 'score').reset_index(drop=True)
top10 = top_score.head(N_SMALL)
print()
print('фінальний топ-10:')
print(top10[['name', 'score', 'wiki', 'n_lang', 'visit', 'hist']].round(3).to_string())

---
## 4. Старт: вокзал, і прив'язка точок до графа

In [ ]:
print('станції у шарі:', len(stations))
print(stations[['name', 'railway', 'building']].dropna(how='all').head(12).to_string())

In [ ]:
st = stations[col(stations, 'name').notna()].copy()
st = st[col(st, 'amenity').ne('bus_station').fillna(True)]          # автовокзали не беремо
st['area'] = st.to_crs(32635).area
city0 = CITY.split(',')[0].strip()
primary = (st['name'].str.contains('Головний|Центральний|Пасажирськ', case=False, regex=True).fillna(False)
           | (st['name'] == city0))
pool = st[primary] if primary.any() else st
station = pool.sort_values('area', ascending=False).head(1)
print('старт:', station['name'].iloc[0])

In [ ]:
G = nx.DiGraph()
for t in nodes.itertuples():
    G.add_node(t.osmid, x=t.x, y=t.y)
for t in edges.itertuples():
    L = float(t.length)
    if G.has_edge(t.u, t.v):
        G[t.u][t.v]['length'] = min(G[t.u][t.v]['length'], L)
    else:
        G.add_edge(t.u, t.v, length=L)
print('граф: %d вузлів, %d ребер' % (G.number_of_nodes(), G.number_of_edges()))

NODES_M = nodes.to_crs(32635)
NODE_XY = np.c_[NODES_M.geometry.x.values, NODES_M.geometry.y.values]
NODE_ID = nodes['osmid'].values

def snap(points_gdf):
    """Найближчий вузол графа до кожної точки. Рахуємо в метричній CRS."""
    pmet = points_gdf.to_crs(32635)
    ids, dist = [], []
    for p in pmet.geometry:
        d2 = (NODE_XY[:, 0] - p.x) ** 2 + (NODE_XY[:, 1] - p.y) ** 2
        k = int(np.argmin(d2))
        ids.append(NODE_ID[k]); dist.append(float(np.sqrt(d2[k])))
    return np.array(ids), np.array(dist)

def make_points(top):
    names = ['СТАРТ: ' + str(station['name'].iloc[0])] + list(top['name'])
    geoms = [station.geometry.iloc[0].representative_point()] + \
            [g.representative_point() for g in top.geometry]
    return gpd.GeoDataFrame({'name': names}, geometry=geoms, crs='EPSG:4326')

### Завдання 4.1 — обов'язкова перевірка

Напишіть `check_snap`, яка повідомляє:

* скільки точок і скільки **різних** вузлів вони зайняли;
* медіану й максимум відстані «точка → вузол»;
* якщо вузлів менше, ніж точок — **назви точок, що злиплись**.

**Контрольна ознака:** на пішохідній мережі різних вузлів має бути **стільки ж,
скільки точок**. Якщо менше — ви взяли автомобільну мережу або місто надто мале;
зафіксуйте це у звіті, бо далі такі пари отримають нульову віддаль.

In [ ]:
def check_snap(points, ids, dist, label):
    n, u = len(ids), len(set(ids))
    k = int(np.argmax(dist))
    print('[%s] точок: %d | різних вузлів: %d' % (label, n, u))
    print('    точка -> вузол: медіана %.0f м, максимум %.0f м («%s»)'
          % (np.median(dist), dist[k], points['name'].iloc[k]))
    if u < n:
        groups = collections.defaultdict(list)
        for i, nd in enumerate(ids):
            groups[nd].append(i)
        print('    ЗЛИПЛИСЬ (нульова віддаль між ними):')
        for nd, idx in groups.items():
            if len(idx) > 1:
                print('      вузол %s: %s' % (nd, ' | '.join(points['name'].iloc[i] for i in idx)))
    else:
        print('    ok: кожна точка має свій вузол')
    return u == n

---
## 5. Матриця віддалей

Дві матриці: **пряма лінія** і **дорогою**. Їхнє відношення — коефіцієнт
обходу — показує, наскільки пряма лінія бреше.

### Завдання 5.1

In [ ]:
def road_matrix(ids):
    """Матриця найкоротших віддалей графом між заданими вузлами."""
    k = len(ids)
    D = np.full((k, k), np.inf)
    for i, s in enumerate(ids):
        lengths = nx.single_source_dijkstra_path_length(G, s, weight='length')
        for j, t in enumerate(ids):
            D[i, j] = lengths.get(t, np.inf)
    np.fill_diagonal(D, 0.0)
    return D

def straight_matrix(points):
    p = points.to_crs(32635).geometry
    XY = np.c_[p.x.values, p.y.values]
    return np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))

def detour_factor(D, E):
    off = ~np.eye(len(D), dtype=bool)
    ok = off & np.isfinite(D) & (E > 1.0)
    return float(np.median(D[ok] / E[ok]))

**Контрольні ознаки — звіртесь, не підганяючи:**

* коефіцієнт обходу для пішохідної мережі у місті зазвичай **1,2–1,6**;
* якщо у вас рівно **1,0** — ви порівнюєте матрицю саму із собою;
* якщо **більше 2** — найімовірніше взято автомобільну мережу з односторонніми
  вулицями, або місто має річку/залізницю, яку нема де перетнути. Друге —
  цікавий результат, перше — помилка. Розберіться, що саме у вас, і напишіть це.

Якщо серед віддалей трапились `inf` — частина графа не зв'язана з рештою.
Не замінюйте `inf` великим числом мовчки: спершу з'ясуйте, які це точки.

In [ ]:
pts10 = make_points(top10)
ids10, dist10 = snap(pts10)
check_snap(pts10, ids10, dist10, 'мала задача')

t0 = time.time(); D10 = road_matrix(ids10); print('матриця за %.3f с' % (time.time() - t0))
E10 = straight_matrix(pts10)
print('недосяжних пар    :', int(np.isinf(D10).sum()))
print('асиметрія         : %.0f м' % float(np.max(np.abs(D10 - D10.T))))
print('коефіцієнт обходу : %.2f' % detour_factor(D10, E10))

> **Асиметрія.** $D_{ij} \\neq D_{ji}$ — не помилка: односторонній рух робить
> шлях туди і назад різним. На пішохідній мережі асиметрія мала, на
> автомобільній — помітна. Подивіться на своє число і поясніть його у звіті.

---
## 6. Задача комівояжера

| метод | що робить | складність |
|---|---|---|
| найближчий сусід | щоразу йде в найближчу невідвідану точку | $O(n^2)$ |
| 2-opt | розвертає ділянки готового маршруту, поки коротшає | $O(n^2)$ за прохід |
| Held–Karp | динамічне програмування за підмножинами — **точний** | $O(2^n n^2)$ часу, $O(2^n n)$ пам'яті |

Held–Karp дано готовим: він не є метою роботи, але без нього ви не дізнаєтесь,
**наскільки** помиляються перші два.

### Завдання 6.1 — реалізуйте перші два методи

In [ ]:
def tour_length(D, tour):
    return float(sum(D[tour[i], tour[i + 1]] for i in range(len(tour) - 1)))

def nearest_neighbour(D):
    """Маршрут 0 -> ... -> 0. Щоразу йдемо в найближчу невідвідану точку."""
    left = set(range(1, len(D)))
    tour = [0]
    while left:
        cur = tour[-1]
        nxt = min(left, key=lambda j: (D[cur, j], j))   # j — для детермінізму при рівних
        tour.append(nxt)
        left.remove(nxt)
    return tour + [0]

def two_opt(D, tour):
    """Беремо маршрут і розвертаємо ділянки, поки довжина зменшується.
    Матриця може бути асиметричною: розворот змінює напрям усіх ребер
    усередині ділянки, тому довжину кандидата рахуємо повністю, а не
    за формулою «мінус два ребра, плюс два ребра»."""
    best = list(tour)
    best_L = tour_length(D, best)
    improved = True
    while improved:
        improved = False
        for i in range(1, len(best) - 2):
            for j in range(i + 1, len(best) - 1):
                cand_t = best[:i] + best[i:j + 1][::-1] + best[j + 1:]
                L = tour_length(D, cand_t)
                if L < best_L - 1e-9:
                    best, best_L, improved = cand_t, L, True
    return best

In [ ]:
def held_karp(D):
    """Точний розв'язок. Вузол 0 — старт і фініш. ДАНО, писати не треба."""
    k = len(D); N = k - 1; INF = float('inf')
    dp = [[INF] * N for _ in range(1 << N)]
    par = [[-1] * N for _ in range(1 << N)]
    for j in range(N):
        dp[1 << j][j] = D[0, j + 1]
    for S in range(1 << N):
        for j in range(N):
            cur = dp[S][j]
            if cur == INF or not (S >> j) & 1:
                continue
            for t in range(N):
                if (S >> t) & 1:
                    continue
                S2 = S | (1 << t); v = cur + D[j + 1, t + 1]
                if v < dp[S2][t]:
                    dp[S2][t] = v; par[S2][t] = j
    full = (1 << N) - 1
    bj = min(range(N), key=lambda j: dp[full][j] + D[j + 1, 0])
    best = dp[full][bj] + D[bj + 1, 0]
    path = []; S = full; j = bj
    while j != -1:
        path.append(j + 1); pj = par[S][j]; S ^= (1 << j); j = pj
    return [0] + path[::-1] + [0], float(best)

### Самоперевірка

Ці твердження мають виконатись для **будь-якого** міста. Якщо хоч одне впало —
шукайте помилку в коді, а не в даних.

In [ ]:
def check_tour(tour, k, label):
    assert tour[0] == 0 and tour[-1] == 0, '%s: маршрут має починатись і закінчуватись на старті' % label
    assert sorted(tour[:-1]) == list(range(k)), '%s: кожна точка рівно один раз' % label
    print('%s: структура маршруту правильна (%d точок)' % (label, k))

nn10 = nearest_neighbour(D10)
op10 = two_opt(D10, nn10)
ex10, L_opt = held_karp(D10)

for t, lbl in [(nn10, 'найближчий сусід'), (op10, '2-opt'), (ex10, 'Held-Karp')]:
    check_tour(t, len(ids10), lbl)

assert tour_length(D10, op10) <= tour_length(D10, nn10) + 1e-6, '2-opt не може бути гіршим за вхідний маршрут'
assert L_opt <= tour_length(D10, op10) + 1e-6, 'точний розв\u2019язок не може бути гіршим за наближений'
print()
print('усі перевірки пройдено')

In [ ]:
rows = []
for name, tour in [('найближчий сусід', nn10), ('2-opt', op10), ('Held-Karp (точний)', ex10)]:
    L = tour_length(D10, tour)
    rows.append({'метод': name, 'довжина, м': round(L),
                 'гірше оптимуму, %': round(100 * (L / L_opt - 1), 2)})
display(pd.DataFrame(rows).set_index('метод'))

print()
print('оптимальний порядок:')
for step, i in enumerate(ex10):
    print('  %2d. %s' % (step, pts10['name'].iloc[i]))
print()
print('довжина: %.2f км' % (L_opt / 1000))

### Завдання 6.2 — велика задача

Спершу оцініть, чи можливий тут точний розв'язок. Порахуйте для
$n = 11, 16, 21, 26, 31$ розмір таблиці Held–Karp ($2^{n-1}\\cdot(n-1)$ клітин)
і скільки це гігабайтів при 8 байтах на клітину.

In [ ]:
rows = []
for n in [11, 16, 21, 26, 31]:
    cells = 2 ** (n - 1) * (n - 1)
    rows.append({'точок n': n, 'клітин': f'{cells:,}'.replace(',', ' '),
                 'ГБ (dp, 8 Б)': round(cells * 8 / 1e9, 3),
                 'ГБ (dp + par)': round(2 * cells * 8 / 1e9, 3)})
growth = pd.DataFrame(rows).set_index('точок n')
display(growth)
HK31_GB = 2 ** 30 * 30 * 8 / 1e9
print('n = 31: %.0f ГБ лише на таблицю dp; з таблицею предків — %.0f ГБ' % (HK31_GB, 2 * HK31_GB))

Подивіться на останній рядок і напишіть у звіті, **чому саме** точний розв'язок
на 30 точках недоступний. Слово «довго» відповіддю не є.

In [ ]:
pts30 = make_points(top_score)
ids30, dist30 = snap(pts30)
check_snap(pts30, ids30, dist30, 'велика задача')

D30 = road_matrix(ids30)
E30 = straight_matrix(pts30)
print('коефіцієнт обходу: %.2f' % detour_factor(D30, E30))

nn30 = nearest_neighbour(D30)
t0 = time.time(); op30 = two_opt(D30, nn30); t_op30 = time.time() - t0
check_tour(op30, len(ids30), '2-opt на 30 точках')

L_nn30, L_op30 = tour_length(D30, nn30), tour_length(D30, op30)
print()
print('найближчий сусід : %8.0f м' % L_nn30)
print('2-opt            : %8.0f м  за %.3f с  (коротше на %.1f%%)'
      % (L_op30, t_op30, 100 * (1 - L_op30 / L_nn30)))
print()
print('ЦЕ — база порівняння для лабораторної роботи № 8.')

---
## 7. Просторова індексація: H3

H3 — ієрархічна сітка шестикутників. Рівень 9 — приблизно 200 м по ребру.

📖 <https://h3geo.org/docs/core-library/restable/>

### Завдання 7.1

Розкладіть **усіх** кандидатів по комірках H3 рівня 9 і побудуйте картограму
щільності. На тій самій карті покажіть межу міста і точки вашого маршруту.

In [ ]:
import h3
from shapely.geometry import Polygon

H3_RES = 9
print('середнє ребро комірки: %.0f м' % h3.average_hexagon_edge_length(H3_RES, unit='m'))

rp = cand.geometry.representative_point()
cand['h3'] = [h3.latlng_to_cell(p.y, p.x, H3_RES) for p in rp]
dens = cand['h3'].value_counts()

# cell_to_boundary -> (lat, lng); shapely хоче (x, y) = (lng, lat)
hexes = gpd.GeoDataFrame({'h3': dens.index, 'n': dens.values},
                         geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)])
                                   for c in dens.index], crs='EPSG:4326')

cum = dens.sort_values(ascending=False).cumsum()
HALF_CELLS = int((cum < len(cand) / 2).sum() + 1)
print('кандидатів %d у %d комірках; половина — у %d комірках (%.0f%% зайнятих)'
      % (len(cand), len(dens), HALF_CELLS, 100 * HALF_CELLS / len(dens)))
print('найщільніша комірка: %d об’єктів' % dens.iloc[0])

fig, ax = plt.subplots(figsize=(9, 8))
boundary.boundary.plot(ax=ax, color='0.5', linewidth=0.8)
hexes.plot(ax=ax, column='n', cmap='YlOrRd', legend=True, edgecolor='0.3', linewidth=0.3,
           legend_kwds={'label': 'об’єктів у комірці H3 r9', 'shrink': 0.6})
pts10.plot(ax=ax, color='#1f4e9c', markersize=25, zorder=5, label='точки маршруту (10 + вокзал)')
pts10.iloc[[0]].plot(ax=ax, color='black', marker='s', markersize=50, zorder=6, label='вокзал')
x0, y0, x1, y1 = pts30.total_bounds if 'pts30' in dir() else pts10.total_bounds
pad = 0.01
ax.set_xlim(x0 - pad, x1 + pad); ax.set_ylim(y0 - pad, y1 + pad)
ax.set_title('Щільність кандидатів, H3 рівень 9 — %s' % CITY.split(',')[0])
ax.legend(loc='lower left'); ax.set_axis_off()
plt.tight_layout(); plt.savefig('lr5_h3.png', dpi=130); plt.show()

# те саме на всю межу міста — щоб видно, що пам'ятки не розкидані рівномірно
fig, ax = plt.subplots(figsize=(7, 6))
boundary.boundary.plot(ax=ax, color='0.5', linewidth=0.8)
hexes.plot(ax=ax, column='n', cmap='YlOrRd', edgecolor='0.3', linewidth=0.3)
ax.set_title('Усе місто'); ax.set_axis_off(); plt.show()

> **Пастка.** `h3.cell_to_boundary` повертає пари **(lat, lng)**, а
> `shapely.Polygon` чекає **(x, y)**, тобто (lng, lat). Переплутаєте — місто
> опиниться в Індійському океані. Це видно одразу, тому дивіться на карту.

### Питання до звіту

У скількох комірках зосереджено половину всіх пам'яток вашого міста?
Що це говорить про форму маршруту?

---
## 8. Просторове поєднання

### Завдання 8.1

Для кожної точки маршруту знайдіть найближчий заклад із шару `amenities`
(`sjoin_nearest`) і виведіть таблицю з відстанню в метрах.

Нагадування: `sjoin_nearest` рахує в одиницях CRS. У градусах результат буде
безглуздим — перепроєктуйте обидва шари в метричну систему.

In [ ]:
AM_OK = ['cafe', 'restaurant', 'drinking_water', 'toilets']
am = amenities[col(amenities, 'amenity').isin(AM_OK).fillna(False)].copy()
print('закладів у шарі amenities:', len(amenities), '| з них', AM_OK, ':', len(am),
      am['amenity'].value_counts().to_dict())
am['geometry'] = am.geometry.representative_point()      # кафе-полігон -> точка
am_m = am[['name', 'amenity', 'geometry']].to_crs(32635)

route_order = ex10[:-1]
pts_m = pts10.iloc[route_order].to_crs(32635).reset_index(drop=True)
pts_m['крок'] = range(len(pts_m))

near = gpd.sjoin_nearest(pts_m, am_m, how='left', distance_col='відстань_м')
near = near[~near.index.duplicated(keep='first')]          # рівновіддалені -> лишаємо один
NEAR = near[['крок', 'name_left', 'amenity', 'name_right', 'відстань_м']].rename(
    columns={'name_left': 'точка маршруту', 'name_right': 'найближчий заклад'})
NEAR['відстань_м'] = NEAR['відстань_м'].round(0).astype(int)
display(NEAR.set_index('крок'))
print('медіана %.0f м, найдальше %.0f м' % (NEAR['відстань_м'].median(), NEAR['відстань_м'].max()))

---
## 9. Маршрут на карті

Лінія маршруту — це **геометрія ребер графа**, а не відрізки між пам'ятками.

### Завдання 9.1

Побудуйте карту, на якій **обидві** лінії: справжній маршрут по ребрах і
прямі відрізки між точками. Різниця має бути видно оком.

In [ ]:
node_xy = nodes.set_index('osmid')[['x', 'y']]

def route_coords(tour, ids):
    """Координати маршруту по ребрах графа у форматі (lat, lon) для folium."""
    coords = []
    for a, b in zip(tour[:-1], tour[1:]):
        try:
            path = nx.shortest_path(G, ids[a], ids[b], weight='length')
        except nx.NetworkXNoPath:
            path = [ids[a], ids[b]]
        for nd in path:
            coords.append((float(node_xy.loc[nd, 'y']), float(node_xy.loc[nd, 'x'])))
    return coords

road = route_coords(ex10, ids10)
straight = [(pts10.geometry.y.iloc[i], pts10.geometry.x.iloc[i]) for i in ex10]   # folium: (lat, lon)
L_straight = tour_length(E10, ex10)

fmap = folium.Map(location=[pts10.geometry.y.mean(), pts10.geometry.x.mean()],
                  zoom_start=15, tiles=None)
folium.TileLayer(tiles=BASEMAP.build_url(), attr=BASEMAP.html_attribution,
                 name='Esri World Gray Canvas').add_to(fmap)

g_road = folium.FeatureGroup(name='маршрут по графу, %.2f км' % (L_opt / 1000))
folium.PolyLine(road, color='#1f4e9c', weight=5, opacity=0.85,
                tooltip='по ребрах графа: %.2f км' % (L_opt / 1000)).add_to(g_road)
g_road.add_to(fmap)

g_line = folium.FeatureGroup(name='прямі відрізки, %.2f км' % (L_straight / 1000))
folium.PolyLine(straight, color='#d7301f', weight=2.5, dash_array='8 6',
                tooltip='напряму: %.2f км' % (L_straight / 1000)).add_to(g_line)
g_line.add_to(fmap)

g_30 = folium.FeatureGroup(name='велика задача: 2-opt, %d точок' % len(ids30), show=False)
folium.PolyLine(route_coords(op30, ids30), color='#2b8a3e', weight=3, opacity=0.7).add_to(g_30)
for i in range(1, len(pts30)):
    folium.CircleMarker((pts30.geometry.y.iloc[i], pts30.geometry.x.iloc[i]), radius=4,
                        color='#2b8a3e', fill=True, tooltip=pts30['name'].iloc[i]).add_to(g_30)
g_30.add_to(fmap)

g_pts = folium.FeatureGroup(name='пам’ятки')
for step, i in enumerate(ex10[:-1]):
    lat, lon = pts10.geometry.y.iloc[i], pts10.geometry.x.iloc[i]
    bg = '#111' if i == 0 else '#1f4e9c'
    label = 'В' if i == 0 else str(step)
    folium.Marker((lat, lon), tooltip='%s. %s' % (label, pts10['name'].iloc[i]),
                  icon=folium.DivIcon(html=(
                      '<div style="background:%s;color:#fff;border-radius:50%%;width:24px;height:24px;'
                      'line-height:24px;text-align:center;font:600 12px sans-serif;'
                      'border:2px solid #fff;box-shadow:0 0 3px #0006">%s</div>' % (bg, label)),
                      icon_size=(24, 24), icon_anchor=(12, 12))).add_to(g_pts)
g_pts.add_to(fmap)

legend = ('<div style="position:fixed;bottom:18px;left:18px;z-index:9999;background:#fff;'
          'padding:8px 12px;border-radius:6px;box-shadow:0 1px 4px #0004;font:13px sans-serif">'
          '<b>%s: вокзал → 10 пам’яток → вокзал</b><br>'
          '<span style="color:#1f4e9c">━━</span> по графу: %.2f км<br>'
          '<span style="color:#d7301f">╍╍</span> прямі відрізки: %.2f км (×%.2f)<br>'
          '<span style="font-size:11px;color:#555">© OpenStreetMap contributors</span></div>'
          % (CITY.split(',')[0], L_opt / 1000, L_straight / 1000, L_opt / L_straight))
fmap.get_root().html.add_child(folium.Element(legend))
folium.LayerControl(collapsed=False).add_to(fmap)
fmap.fit_bounds([[min(p[0] for p in road), min(p[1] for p in road)],
                 [max(p[0] for p in road), max(p[1] for p in road)]])
fmap.save('lr5_маршрут.html')
print('карта: lr5_маршрут.html | по графу %.2f км, прямими %.2f км' % (L_opt / 1000, L_straight / 1000))
fmap

---
## 10. Результати для лабораторної роботи № 8

Збережіть файл, який ЛР8 прочитає і спробує побити.

In [ ]:
result = {
    'студент': STUDENT_NAME,
    'номер': LIST_NUMBER,
    'місто': CITY,
    'мала_задача': {
        'точок': len(ids10),
        'оптимум_м': round(L_opt, 1),
        'оптимальний_порядок': [str(pts10['name'].iloc[i]) for i in ex10],
        'найближчий_сусід_м': round(tour_length(D10, nn10), 1),
        'двоопт_м': round(tour_length(D10, op10), 1),
    },
    'велика_задача': {
        'точок': len(ids30),
        'найближчий_сусід_м': round(L_nn30, 1),
        'двоопт_м': round(L_op30, 1),
        'двоопт_с': round(t_op30, 4),
        'точки': [{'name': str(pts30['name'].iloc[i]),
                   'lat': float(pts30.geometry.y.iloc[i]),
                   'lon': float(pts30.geometry.x.iloc[i])} for i in range(len(pts30))],
        'матриця_віддалей_м': [[None if not np.isfinite(v) else round(float(v), 1)
                                for v in row] for row in D30],
    },
}
with open('lr5_результати.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=1)
print('збережено lr5_результати.json (%.1f КБ)' % (os.path.getsize('lr5_результати.json') / 1024))

---
## 11. Звіт

Відповіді пишіть просто в наступній комірці, числами зі своїх обчислень.

1. **Скільки об'єктів ви відкинули** на кроці 2 і за яким правилом? Чи погодились
   би ви віддати цей фільтр замовнику без пояснень?
2. **Які сигнали ввійшли у ваш рейтинг і з якими вагами?** Чому саме так?
3. **Скільки об'єктів збіглося** у ваших двох рейтингах із десяти? Що це
   означає для слова «найкращі»?
4. **Який у вас коефіцієнт обходу** і чи типовий він? Якщо ні — що в місті
   його пояснює?
5. **На скільки відсотків** найближчий сусід гірший за точний розв'язок?
   Чи здивувало вас це число?
6. **Чому точний розв'язок на 30 точках недоступний?** Назвіть конкретну
   величину, яка цьому заважає.
7. **Що у ваших даних виявилось неправильним** — дублікати, пропущені назви,
   об'єкти не на місці? Як ви з цим вчинили?

### Декларація про використання штучного інтелекту

Зазначте, чи користувались ШІ-помічником, у яких саме місцях і як перевіряли
отриманий код. Використання дозволене; **неперевірене** використання — ні.

In [ ]:
_E = straight_matrix(pts30)[0]
FAR_KM, FAR_NAME = _E.max() / 1000, pts30['name'].iloc[int(_E.argmax())]
nn_gap = 100 * (tour_length(D10, nn10) / L_opt - 1)
op_gap = 100 * (tour_length(D10, op10) / L_opt - 1)
df10, df30 = detour_factor(D10, E10), detour_factor(D30, E30)
asym10 = float(np.max(np.abs(D10 - D10.T)))
snap_ok10 = len(set(ids10)) == len(ids10)
snap_ok30 = len(set(ids30)) == len(ids30)
top_wiki = int(top10['wiki'].sum())
only_naive = ', '.join(n for n in top_naive['name'] if n not in set(top10['name']))
only_score = ', '.join(n for n in top10['name'] if n not in set(top_naive['name']))

REPORT = f'''
Місто: {CITY}. За варіантом ({LIST_NUMBER}) — {CITIES[(LIST_NUMBER - 1) % len(CITIES)]};
{"місто призначене викладачем вручну." if CITY_OVERRIDE else ""}
Доступ до даних: {TIMING}

1. У шарі heritage {len(heritage)} об'єктів, кандидатів лишилось {len(cand_raw)}
   (після злиття дублікатів — {len(cand)}). Відкинуто: {DROPPED}.
   Правило: historic заповнений АБО tourism ∈ {TOURISM_OK}; без назви — геть
   (у маршрут не поставиш те, що не можеш назвати); придорожні хрести, межові
   камені й меморіальні дошки — геть, бо це не мета пішої прогулянки.
   Без пояснень замовнику не віддав би: «без назви» відкидає і справжні пам'ятки,
   яким просто ніхто не вписав name.

2. score = 0.35·wiki + 0.30·rank(n_lang) + 0.15·museum/gallery + 0.10·historic
   + 0.10·rank(площа). wiki — найсильніший: зовнішнє визнання, а не рішення
   картографа. Переклади назви — попит з боку іноземців, але корелюють з wiki,
   тому трохи менша вага. Музей/галерея — об'єкт для відвідування. Площа — найслабша,
   бо великим буває і склад. Перевірка стійкості: при випадковій зміні ваг на ±50%
   топ-10 зберігає в середньому {STAB.mean():.1f} з 10 (мінімум {STAB.min()}).
   У фінальному топ-10 {top_wiki} з 10 мають wikidata/wikipedia.

3. Збіглося {overlap} з {N_SMALL}. Лише за площею: {only_naive}.
   Лише за моїм критерієм: {only_score}.
   Площа тягне нагору великі споруди (форт, фортечні вежі), байдуже, чи є там що
   дивитись; мій критерій — музеї з wiki-статтями. Обидва списки
   правильно пораховані з тих самих даних, тож «найкращі» — це не властивість
   даних, а рішення аналітика, яке треба називати й захищати.

4. Коефіцієнт обходу {df10:.2f} (10 точок) і {df30:.2f} (30 точок);
   асиметрія до {asym10:.0f} м. {"Типово для пішохідної мережі." if 1.15 <= df10 <= 1.6 and WALK_OK else
   f"Мережа у знімку автомобільна (0 ребер footway/pedestrian/path): односторонні вулиці дають асиметрію, пішохідні вулиці старого міста з графа випали (медіана точка→вузол {np.median(dist10):.0f} м, максимум {dist10.max():.0f} м), тому шлях між сусідніми пам'ятками робить гак кварталом."}

5. Найближчий сусід гірший за точний розв'язок на {nn_gap:.1f}%, 2-opt — на {op_gap:.1f}%.
   Оптимум {L_opt/1000:.2f} км. На 30 точках 2-opt скоротив NN на
   {100 * (1 - L_op30 / L_nn30):.1f}% ({L_nn30/1000:.2f} → {L_op30/1000:.2f} км) за {t_op30:.2f} с.

6. Таблиця Held–Karp для n = 31 має 2^30·30 ≈ 3.2·10^10 клітин, тобто
   ≈ {HK31_GB:.0f} ГБ при 8 байтах (з таблицею предків ≈ {2*HK31_GB:.0f} ГБ). Це в десятки
   разів більше за оперативну пам'ять ноутбука. Обмеження — пам'ять, а не час.

7. Проблеми даних:
   • {"усі кандидати — полігони будівель, точкових пам'ятників немає: знімок містить лише шар buildings, а не heritage" if (cand.geometry.geom_type == "Point").sum() == 0 else "є і точки, і полігони — ок"};
   • мережа {"автомобільна, хоча тека називається walk" if not WALK_OK else "пішохідна"};
   • прив'язка: {len(ids10)} точок → {len(set(ids10))} вузлів, {len(ids30)} точок → {len(set(ids30))} вузлів
     {"(злиплі пари отримали нульову віддаль — перелік вище в кроці 5/6.2)" if not (snap_ok10 and snap_ok30) else "(злиплих немає)"};
   • дублікати < 60 м: злито {len(DUPES)} ({"; ".join(f"{a} / {b}: {w}" for a, b, _, w in DUPES) or "немає"}),
     лишено як різні {len(NEAR_KEPT)} пар (сусідні, але окремі об'єкти);
   • {DROPPED["без назви"]} об'єктів без назви;
   • велика задача: {len(ids30)-1} точок дають {L_op30/1000:.1f} км; найдальша від вокзалу —
     «{FAR_NAME}» ({FAR_KM:.1f} км по прямій), тож топ-30 за рейтингом — не піша
     прогулянка на день: критерій не знає нічого про відстань.

H3 (рівень 9, ребро ≈ 200 м): {len(cand)} кандидатів у {len(dens)} комірках, половина з них —
лише в {HALF_CELLS} комірках. Маршрут топ-10 — це компактна петля в історичному центрі
плюс дорога від вокзалу й назад, а для топ-30 ключове — скільки «острівців» щільності
в місті, бо перехід між ними і є основна довжина маршруту.
   Висновок, якого не було в завданні: помилка в даних не дає помилки в коді —
   автомобільна мережа, злиплі вузли і дублікати проходять усі assert; їх видно лише
   з явних перевірок (частка footway, check_snap, close_pairs).

Декларація про використання ШІ: код незаповнених комірок написано з допомогою
Claude (Anthropic). Перевірено: самоперевірки check_tour і assert ноутбука,
2-opt ≤ NN ≤ …, Held–Karp ≤ 2-opt, контрольні ознаки (кількість вузлів,
коефіцієнт обходу, тип геометрій), карта переглянута очима.

© OpenStreetMap contributors
'''
print(REPORT)